<center><img src="img/header_gr13_outlier_anomalies.jpg"></center>

# Airbnb Listings
The dataset contains information on **17,432 Airbnb listings in Bangkok**. Each row represents a listing with details such as coordinates, neighborhood, host id, price per night, number of reviews, and so on. 

[Source: InsideAirbnb](http://insideairbnb.com)

## Data Dictionary

| Column                            | Explanation                                                                                                                                                                                        |
| --------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| id                                | Airbnb's unique identifier for the listing                                                                                                                                                         |
| name                              |                                                                                                                                                                                                    |
| host\_id                          |                                                                                                                                                                                                    |
| host\_name                        |                                                                                                                                                                                                    |
| neighbourhood\_group              | The neighbourhood group as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                        |
| neighbourhood                     | The neighbourhood as geocoded using the latitude and longitude against neighborhoods as defined by open or public digital shapefiles.                                                              |
| latitude                          | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| longitude                         | Uses the World Geodetic System (WGS84) projection for latitude and longitude.                                                                                                                      |
| room\_type                        |                                                                                                                                                                                                    |
| price                             | daily price in local currency. Note, $ sign may be used despite locale                                                                                                                             |
| minimum\_nights                   | minimum number of night stay for the listing (calendar rules may be different)                                                                                                                     |
| number\_of\_reviews               | The number of reviews the listing has                                                                                                                                                              |
| last\_review                      | The date of the last/newest review                                                                                                                                                                 |
| calculated\_host\_listings\_count | The number of listings the host has in the current scrape, in the city/region geography.                                                                                                           |
| availability\_365                 | avaliability\_x. The availability of the listing x days in the future as determined by the calendar. Note a listing may be available because it has been booked by a guest or blocked by the host. |
| number\_of\_reviews\_ltm          | The number of reviews the listing has (in the last 12 months)                                                                                                                                      |
| license                           |                                                                                                                                                                                                    |

The data was compiled by [InsideAirbnb](http://insideairbnb.com) between October and November 2021.

[Source](http://insideairbnb.com/get-the-data.html) and [license](https://creativecommons.org/licenses/by/4.0/) of dataset. 

#     Scenario
You are part of the Trust & Safety Team for a regional travel agency. Your company is launching a "Verified Luxury" program in Bangkok. Before the launch, you must audit the 17,432 listings to ensure that properties claiming to be high-end are legitimate and that no "ghost listings" (fake listings created to scam users) have entered the dataset.

**Background:**
Simple statistical methods like Z-Score or IQR only look at Univariate data (one variable). For example, a price of 20,000 THB might look like an outlier on its own. However, if that listing is a "Private Villa" with 100 reviews and 365 days of availability, it is a perfectly normal luxury listing.

* The problem arises with Multivariate Anomalies. These are listings that look normal in each individual column but are "weird" when you combine them.

* Example: A listing priced at a very low 500 THB (normal) but listed as an "Entire Home" (unusual for that price) with 0 reviews and 0 availability (suspicious).

* Simple filters won't catch this, but Isolation Forest and One-Class SVM will.

**Objective:**
Your objective is to detect Complex Outliers that threaten the integrity of the marketplace. You are trying to catch:

  1. Mislabeled Listings: Properties that claim a high-end room_type but have a price too low to be realistic, or vice versa.

  2. Inactive/Zombie Listings: Listings with high availability_365 but zero number_of_reviews and high minimum_nights. These often represent data entry errors or abandoned accounts that clutter the platform.

  3. The "Hidden" Outlier: Identifying listings that exist in "lonely" parts of the data space—where the combination of price, availability, and guest requirements doesn't match any known successful business model in Bangkok.

<center><img src="img/multivariate.jpg"></center>

# CODE

In [1]:
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import numpy as np

# Load and prepare data
df = pd.read_csv("listings_bangkok.csv")
features = ['price', 'minimum_nights', 'number_of_reviews', 'availability_365']
data = df[features].dropna()
print(data.head())


   price  minimum_nights  number_of_reviews  availability_365
0   1821               3                 65               362
1   1260               1                  0               358
2   1159              28                 52               364
3    800              60                  0               365
4    800               5                  1               365


<center><img src="img/demo/isolation_forest.jpg"></center>

<h1 style="font-weight: bold;"> <center>Isolation Forest</center></h1> 
This algorithm works by "isolating" data points.

In [2]:
# Initialize the model
# contamination=0.05 means we expect roughly 5% of the data to be outliers
iso_forest = IsolationForest(contamination=0.05, random_state=42)

# Fit the model and predict
# 1 = normal, -1 = outlier
data['anomaly'] = iso_forest.fit_predict(data)

# Show results
outliers = data[data['anomaly'] == -1]
print(f"Number of outliers detected: {len(outliers)}")
print(outliers.head())

Number of outliers detected: 872
    price  minimum_nights  number_of_reviews  availability_365  anomaly
9    2175               2                208                 0       -1
14   5431              28                148               362       -1
21   1733               3                205               239       -1
27   1159              30                194               365       -1
29    861             358                 56               365       -1


<center><img src="img/demo/ocSVM.jpg"></center>

<h1 style="font-weight: bold;"> <center>One-Class SVM</center></h1> 
A Support Vector Machine (SVM) usually separates two groups. A One-Class SVM is a special version that only looks at one group: the "normal" data. It draws a Decision Boundary (a mathematical "fence") around the densest area of data. Anything that falls outside this fence is labeled an anomaly.